In [11]:
import os
import torch
from torch.utils.data import Dataset
import numpy as np

class ProstateCancerDataset(Dataset):
    def __init__(self, path_to_dataset):
        self.path_to_dataset = path_to_dataset
        self.patients = os.listdir(path_to_dataset)

    def __getitem__(self, idx):
        patient = self.patients[idx]

        # gather images for ADC, CDIs, and DWI modalities
        # NOTE: limiting number of slices to maximum 19
        adc = np.load(os.path.join(self.path_to_dataset, f"{patient:04}\images\ADC.npy"))[:, :, :19]
        cdis = np.load(os.path.join(self.path_to_dataset, f"{patient:04}\images\CDIs.npy"))[:, :, :19]
        dwi = np.load(os.path.join(self.path_to_dataset, f"{patient:04}\images\DWI.npy"))[:, :, :, :19]

        # transpose dwi into appropriate format for reshaping
        dwi_swap = np.transpose(dwi, (1, 2, 3, 0))
        dwi_reshape = np.reshape(dwi_swap, (128, 84, dwi_swap.shape[2]*dwi_swap.shape[3]))

        # append adc onto dwi, and then append cdis to stack all together
        temp_array = np.append(dwi_reshape, adc, axis=2)
        img_stacked = np.append(temp_array, cdis, axis=2)

        # create label map for location of prostate and lesion
        prostate_mask = np.load(os.path.join(self.path_to_dataset, f"{patient:04}\prostate_masks\ADC.npy"))[:, :, :19]
        lesion_mask = np.load(os.path.join(self.path_to_dataset, f"{patient:04}\lesion_masks\ADC.npy"))[:, :, :19]
        cancer_mask = prostate_mask * lesion_mask

        # convert image and mask to tensors
        img_tensor = torch.from_numpy(np.transpose(img_stacked, (2, 0, 1))).float()
        cancer_tensor = torch.from_numpy(np.transpose(cancer_mask, (2, 0, 1))).float()

        return img_tensor, cancer_tensor

    def __len__(self):
        return len(self.patients)

In [12]:
import torch
import torch.nn as  nn
import torch.nn.functional as F


class Bottleneck(nn.Module):
    expansion = 4
    def __init__(self, in_channels, out_channels, i_downsample=None, stride=1):
        super(Bottleneck, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0)
        self.batch_norm1 = nn.BatchNorm2d(out_channels)
        
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.batch_norm2 = nn.BatchNorm2d(out_channels)
        
        self.conv3 = nn.Conv2d(out_channels, out_channels*self.expansion, kernel_size=1, stride=1, padding=0)
        self.batch_norm3 = nn.BatchNorm2d(out_channels*self.expansion)
        
        self.i_downsample = i_downsample
        self.stride = stride
        self.relu = nn.ReLU()
        
    def forward(self, x):
        identity = x.clone()
        x = self.relu(self.batch_norm1(self.conv1(x)))
        
        x = self.relu(self.batch_norm2(self.conv2(x)))
        
        x = self.conv3(x)
        x = self.batch_norm3(x)
        
        #downsample if needed
        if self.i_downsample is not None:
            identity = self.i_downsample(identity)
        #add identity
        x+=identity
        x=self.relu(x)
        
        return x

class Block(nn.Module):
    expansion = 1
    def __init__(self, in_channels, out_channels, i_downsample=None, stride=1):
        super(Block, self).__init__()
       

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, stride=stride, bias=False)
        self.batch_norm1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, stride=stride, bias=False)
        self.batch_norm2 = nn.BatchNorm2d(out_channels)

        self.i_downsample = i_downsample
        self.stride = stride
        self.relu = nn.ReLU()

    def forward(self, x):
      identity = x.clone()

      x = self.relu(self.batch_norm2(self.conv1(x)))
      x = self.batch_norm2(self.conv2(x))

      if self.i_downsample is not None:
          identity = self.i_downsample(identity)
      print(x.shape)
      print(identity.shape)
      x += identity
      x = self.relu(x)
      return x


        
        
class ResNet(nn.Module):
    def __init__(self, ResBlock, layer_list, num_classes, num_channels=3):
        super(ResNet, self).__init__()
        self.in_channels = 64
        
        self.conv1 = nn.Conv2d(num_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.batch_norm1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        self.max_pool = nn.MaxPool2d(kernel_size = 3, stride=2, padding=1)
        
        self.layer1 = self._make_layer(ResBlock, layer_list[0], planes=64)
        self.layer2 = self._make_layer(ResBlock, layer_list[1], planes=128, stride=2)
        self.layer3 = self._make_layer(ResBlock, layer_list[2], planes=256, stride=2)
        self.layer4 = self._make_layer(ResBlock, layer_list[3], planes=512, stride=2)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(512*ResBlock.expansion, num_classes)
        
    def forward(self, x):
        x = self.relu(self.batch_norm1(self.conv1(x)))
        x = self.max_pool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        x = self.avgpool(x)
        x = x.reshape(x.shape[0], -1)
        x = self.fc(x)
        
        return x
        
    def _make_layer(self, ResBlock, blocks, planes, stride=1):
        ii_downsample = None
        layers = []
        
        if stride != 1 or self.in_channels != planes*ResBlock.expansion:
            ii_downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, planes*ResBlock.expansion, kernel_size=1, stride=stride),
                nn.BatchNorm2d(planes*ResBlock.expansion)
            )
            
        layers.append(ResBlock(self.in_channels, planes, i_downsample=ii_downsample, stride=stride))
        self.in_channels = planes*ResBlock.expansion
        
        for i in range(blocks-1):
            layers.append(ResBlock(self.in_channels, planes))
            
        return nn.Sequential(*layers)

        
        
def ResNet50(num_classes, channels=3):
    return ResNet(Bottleneck, [3,4,6,3], num_classes, channels)
    
def ResNet101(num_classes, channels=3):
    return ResNet(Bottleneck, [3,4,23,3], num_classes, channels)

def ResNet152(num_classes, channels=3):
    return ResNet(Bottleneck, [3,8,36,3], num_classes, channels)

In [ ]:
import torchvision.models as models

model = ResNet50(2, channels=95)
num_ftrs = model.fc.in_features
model.fc = torch.nn.Linear(num_ftrs, 1)

# move model to GPU if available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
from torch.utils.data import DataLoader

dataset = ProstateCancerDataset('data\PreparedPROSTATExDataset\data')
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# instantiate ResNet50 model and optimizer here
num_epochs = 10
for epoch in range(num_epochs):
    running_loss = 0.0
    for i, data in enumerate(dataloader, 0):
        inputs, labels = data
        print(f"{inputs.shape}, {labels.shape}")
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 10 == 9:    # print every 10 mini-batches
            print('[%d, %5d] loss: %.3f' %
                  (epoch + 1, i + 1, running_loss / 10))
            running_loss = 0.0
        

In [ ]:
data\PreparedPROSTATExDataset\data\ProstateX-0004\images\ADC.npy

In [ ]:
# gather images for ADC, CDIs, and DWI modalities
adc = np.load('data\PreparedPROSTATExDataset\data\ProstateX-0001\images\ADC.npy')
cdis = np.load('data\PreparedPROSTATExDataset\data\ProstateX-0001\images\CDIs.npy')
dwi = np.load('data\PreparedPROSTATExDataset\data\ProstateX-0001\images\DWI.npy')

# stack images and channels into 2D, reshape into three dimensions
total_slice = adc.shape[2] + cdis.shape[2] + dwi.shape[2]
dwi_reshape = np.reshape(dwi, (128, 84, dwi.shape[0]*dwi.shape[3]))
# img_stacked = np.stack((adc, cdis, dwi_reshape), axis=3)
# img_reshape = np.reshape(img_stacked, (128, 84, total_slice))[:, :, 0:min(adc.shape[2], cdis.shape[2], dwi.shape[2])]




dwi_swap = np.transpose(dwi, (1, 2, 3, 0))
dwi_reshape = np.reshape(dwi_swap, (128, 84, dwi_swap.shape[2]*dwi_swap.shape[3]))

new_array = np.append(dwi_reshape, adc, axis=2)
img_stacked = np.append(new_array, cdis, axis=2)

img_stacked.shape

plt.imshow(img_stacked[ :, :, 1], cmap='gray')
plt.show()


In [ ]:
# play around with shape of labels -> output has to be specific thing
# 